# 🎙️ VoiceBatch Studio v0.0 - [Super Fast Mode]
Isme Silence Remover, Word Counter aur High-Speed Upload shamil hai.

In [ ]:
# @title 🛠️ Step 1: High-Speed Setup
import os
from google.colab import drive
print("⏳ Setup ho raha hai... Thoda intezar karein.")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ Setup taiyar hai!")

In [ ]:
# @title 🚀 Step 2: Launch Super Fast Studio
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ Model load ho raha hai...")
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def voice_batch_engine(text, audio_sample, remove_silence):
    try:
        if not audio_sample or not text: return None
        
        parts = re.split(r'(?<=[।?!])\s+', text)
        combined_audio = []
        
        for p in parts:
            if len(p.strip()) < 2: continue
            wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
            combined_audio.append(np.array(wav))
        
        final_wav = np.concatenate(combined_audio)
        
        # Silence Remover Logic
        if remove_silence:
            final_wav, _ = librosa.effects.trim(final_wav, top_db=20)
            
        out_path = "outputs/VoiceBatch_Final.wav"
        sf.write(out_path, final_wav, 24000)
        return out_path
    except Exception as e:
        return f"Galti: {str(e)}"

def count_words(text):
    return f"Shabd: {len(text.split())}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio v0.0")
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label="Yahan apni kahani likhein", lines=10)
            word_count = gr.Label(value="Shabd: 0", label="Word Counter")
            txt.change(count_words, inputs=[txt], outputs=[word_count])
            
            # High Speed Audio Upload
            smp = gr.Audio(label="Apna voice sample upload karein", type='filepath')
            
            sil_btn = gr.Checkbox(label="Sannata hatayein (Silence Remover)", value=True)
            btn = gr.Button("Audio Banayein ⚡", variant="primary")
        
        with gr.Column():
            out = gr.Audio(label="Taiyar Audio")
            
    btn.click(voice_batch_engine, [txt, smp, sil_btn], out)

demo.launch(share=True, debug=True)